# Paper figure: environmental properties and tilt
One two-panel figure: **planetary beta** and **log10 environmental PV-gradient magnitude**, each versus measured tilt distance in kilometres. Red = AE, blue = CE. Lines are eddy-day medians; light shading is the interquartile range, not a confidence interval.

These are **unadjusted associations**. Beta co-varies with latitude; the total PV-gradient relationship weakened after adjustment in the broader notebook. This figure does not attribute causation or independent explanatory power to either predictor. No additional mechanisms or adjusted models are shown here.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd().resolve()
ANALYSIS=next((p for p in (HERE,*HERE.parents) if (p/'seacofs_tilt_tools.py').exists()),None)
if ANALYSIS is None: raise FileNotFoundError('Launch inside seacofs_eddy_tilt_analysis')
WORK=ANALYSIS/'eddy_env_tilt_controls'
for p in (ANALYSIS,WORK):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
import seacofs_tilt_tools as tilt
import paper_control_tools as paper
paths=tilt.Paths()
N_BINS=8
MIN_EDDIES_PER_BIN=20
SAVE=True
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_ROOT=WORK/'paper_figures'/RUN_ID
surface,_=tilt.load_tilt_tables(paths)
print('Loaded',len(surface),'eddy-days from',surface.Eddy.nunique(),'eddies')

## Load the existing environmental PV cache
Use the same definition as the broader environment notebook: Gaussian footprint at fraction 1, nonlinear averaging, and signed peak-|vorticity| through 1000 m. `PV_grad_mag` is the environmental planetary-plus-topographic term, excluding the internal eddy-vorticity-gradient contribution. Settings are checked by the established cache loader; no cache is rebuilt.

Only beta and PV magnitude are joined onto the current Eddy-Day tilt table. Both panels use the same finite observations. PV magnitudes ≤ 0 cannot be logged and are excluded from both panels. Beta is df/dy, **not** beta/h. The log axis explicitly normalises by 1 m⁻² s⁻¹.

In [ ]:
PV_CACHE=tilt.DEFAULT_SURFACE_PV_CACHE
ELLIPSE_FRAC=1.
USE_MAX_ABS_VORTICITY=True
PV_MAX_DEPTH_M=1000
if not PV_CACHE.exists(): raise FileNotFoundError(f'Existing surface PV cache required: {PV_CACHE}')
pv=tilt.add_pv_gradient_terms(core_mean=True,frac=ELLIPSE_FRAC,surface_method='esp_gaussian',
                             averaging='nonlinear',use_max_abs_w=USE_MAX_ABS_VORTICITY,
                             max_depth_m=PV_MAX_DEPTH_M,use_cache=True,cache_path=PV_CACHE)
environment,availability=paper.environment_data(surface,pv)
counts=environment.groupby('Cyc',as_index=False).agg(eddy_days=('Day','size'),eddies=('Eddy','nunique'))
display(availability)
display(counts)
if environment.empty: raise ValueError('No matched finite beta/PV/tilt observations')

## Figure — beta and environmental PV-gradient magnitude
Shared quantile bins within each panel; bins with too few distinct eddies remain gaps. The figure contains only median curves, IQR shading, units and polarity labels.

In [ ]:
fig_environment,environment_bins=paper.environment_figure(environment,bins=N_BINS,min_eddies=MIN_EDDIES_PER_BIN)
plt.show()

## Supporting correlations and export
The small table below uses one median pair per eddy, separately for AE and CE. It is descriptive and is not placed on the figure. Save the full bin counts alongside the PDF/PNG.

In [ ]:
from scipy.stats import spearmanr
correlations=[]
for metric in ['beta','log10_PV_grad_mag']:
    for cyc,part in environment.groupby('Cyc'):
        eddy=part.groupby('Eddy')[[metric,'TiltDis']].median()
        rho=spearmanr(eddy[metric],eddy.TiltDis).statistic if len(eddy)>2 and eddy[metric].nunique()>1 and eddy.TiltDis.nunique()>1 else np.nan
        correlations.append(dict(metric=metric,Cyc=cyc,eddies=len(eddy),rho=rho))
correlations=pd.DataFrame(correlations)
display(correlations)
if SAVE:
    output=OUTPUT_ROOT/'environment'
    paper.save_figure(fig_environment,output,'environment_controls')
    for name,table in {'availability':availability,'sample_counts':counts,'median_bins':environment_bins,
                       'eddy_equal_correlations':correlations}.items():table.to_csv(output/(name+'.csv'),index=False)
    settings=dict(n_bins=N_BINS,min_eddies_per_bin=MIN_EDDIES_PER_BIN,ellipse_frac=ELLIPSE_FRAC,
                  use_max_abs_vorticity=USE_MAX_ABS_VORTICITY,pv_max_depth_m=PV_MAX_DEPTH_M,
                  surface_method='esp_gaussian',averaging='nonlinear',curves='eddy-day medians; IQR shading',
                  pv_units='m^-2 s^-1',pv_transform='log10',beta_units='m^-1 s^-1')
    paper.save_metadata(output,settings,[paths.eddies,paths.tilt,PV_CACHE,WORK/'paper_control_tools.py',WORK/'paper_environment_controls.ipynb'])
    print('Paper figure and supporting tables:',output)